# Data Cleaning & Feature Engineering

## Overview

This notebook prepares the localized ChriLy datasets for downstream SQL analytics and Power BI visualization.

The objectives are to:

- Correct data quality issues identified during the audit.
- Preserve business logic while cleaning invalid records.
- Engineer meaningful business features.
- Export cleaned datasets.

Unlike many BI projects, the datasets remain **normalized**. Table joins and KPI calculations are intentionally deferred to SQL in order to demonstrate relational database and analytical query skills.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

RAW_PATH = Path("../data/morocco")
OUTPUT_PATH = Path("../data/clean")

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

# Load datasets

In [2]:
customers = pd.read_csv(
    RAW_PATH / "chrily_customers.csv"
)

geolocation = pd.read_csv(
    RAW_PATH / "chrily_geolocation.csv"
)

order_items = pd.read_csv(
    RAW_PATH / "chrily_order_items.csv"
)

order_payments = pd.read_csv(
    RAW_PATH / "chrily_order_payments.csv"
)

order_reviews = pd.read_csv(
    RAW_PATH / "chrily_order_reviews.csv"
)

orders = pd.read_csv(
    RAW_PATH / "chrily_orders.csv"
)

products = pd.read_csv(
    RAW_PATH / "chrily_products.csv"
)

sellers = pd.read_csv(
    RAW_PATH / "chrily_sellers.csv"
)

category_translation = pd.read_csv(
    RAW_PATH / "chrily_category_translation.csv"
)

In [3]:
overview = pd.DataFrame({

    "Dataset":[

        "Customers",
        "Geolocation",
        "Order Items",
        "Payments",
        "Reviews",
        "Orders",
        "Products",
        "Sellers",
        "Category Translation"

    ],

    "Rows":[

        len(customers),
        len(geolocation),
        len(order_items),
        len(order_payments),
        len(order_reviews),
        len(orders),
        len(products),
        len(sellers),
        len(category_translation)

    ],

    "Columns":[

        customers.shape[1],
        geolocation.shape[1],
        order_items.shape[1],
        order_payments.shape[1],
        order_reviews.shape[1],
        orders.shape[1],
        products.shape[1],
        sellers.shape[1],
        category_translation.shape[1]

    ]

})

overview

,Dataset,Rows,Columns
0,Customers,99441,6
1,Geolocation,1000163,5
2,Order Items,112650,9
3,Payments,103886,5
4,Reviews,99224,7
5,Orders,99441,14
6,Products,32951,9
7,Sellers,3095,4
8,Category Translation,71,2


# Data Type Conversion

In [4]:
orders_dates = [

    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"

]

for column in orders_dates:

    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

review_dates = [

    "review_creation_date",
    "review_answer_timestamp"

]

for column in review_dates:

    order_reviews[column] = pd.to_datetime(
        order_reviews[column],
        errors="coerce"
    )

print("✓ Datetime conversion completed.")

✓ Datetime conversion completed.


# Duplicate Detection

In [6]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}


In [7]:
duplicate_summary = []

for name, df in datasets.items():

    duplicate_summary.append({
        "Dataset": name,
        "Exact Duplicate Rows": df.duplicated().sum()
    })

duplicate_summary = pd.DataFrame(duplicate_summary)

duplicate_summary

,Dataset,Exact Duplicate Rows
0,customers,0
1,geolocation,1000122
2,order_items,0
3,order_payments,0
4,order_reviews,0
5,orders,0
6,products,0
7,sellers,0
8,category_translation,0


The duplicate audit shows that the datasets do not contain problematic duplicate records requiring removal.

# Missing Value Treatment

In [8]:
products[
    [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].isna().sum()

product_weight_g     2
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64

In [9]:
dimension_columns = [

    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"

]

for column in dimension_columns:

    median = products[column].median()

    products[column] = products[column].fillna(median)

print("✓ Missing product dimensions filled with median.")

✓ Missing product dimensions filled with median.


In [10]:
products["product_category_name"] = products[
    "product_category_name"
].fillna("Unknown")

print("✓ Missing product categories filled.")

✓ Missing product categories filled.


In [11]:
orders.loc[
    orders["order_delivered_customer_date"].isna(),
    "order_status"
].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [12]:
order_reviews[
    [
        "review_comment_title",
        "review_comment_message"
    ]
].isna().sum()

review_comment_title      87656
review_comment_message    58247
dtype: int64

Review comments are optional customer feedback.

# Invalid Value Treatment

In [13]:
order_payments.loc[
    order_payments["payment_value"] <= 0,
    [
        "order_id",
        "payment_type",
        "payment_installments",
        "payment_value"
    ]
]

,order_id,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,Gift Voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,Gift Voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,Gift Voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,Other,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,Other,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,Gift Voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,Gift Voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,Other,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,Gift Voucher,1,0.0


In [14]:
before = len(order_payments)

order_payments = order_payments[
    order_payments["payment_value"] > 0
].copy()

removed_payment_rows = before - len(order_payments)

print(f"Removed {removed_payment_rows} invalid payment records.")

Removed 9 invalid payment records.


In [15]:
order_payments.loc[
    order_payments["payment_installments"] < 1,
    [
        "order_id",
        "payment_type",
        "payment_installments"
    ]
]

,order_id,payment_type,payment_installments
46982,744bade1fcf9ff3f31d860ace076d422,Bank Card,0
79014,1a57108394169c0b47d8f876acc9ba2d,Bank Card,0


In [16]:
before = len(order_payments)

order_payments = order_payments[
    order_payments["payment_installments"] >= 1
].copy()

removed_installments = before - len(order_payments)

print(f"Removed {removed_installments} invalid installment records.")

Removed 2 invalid installment records.


In [17]:
products.loc[
    products["product_weight_g"] <= 0,
    [
        "product_id",
        "product_weight_g"
    ]
]

,product_id,product_weight_g
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,0.0
13683,8038040ee2a71048d4bdbbdc985b69ab,0.0
14997,36ba42dd187055e1fbe943b2d11430ca,0.0
32079,e673e90efa65a5409ff4196c038bb5af,0.0


In [18]:
median_weight = products.loc[
    products["product_weight_g"] > 0,
    "product_weight_g"
].median()

products.loc[
    products["product_weight_g"] <= 0,
    "product_weight_g"
] = median_weight

print("Invalid product weights replaced with the median.")

Invalid product weights replaced with the median.


In [19]:
validation = pd.DataFrame({

    "Check":[

        "Payment Value > 0",
        "Payment Installments >= 1",
        "Product Weight > 0"

    ],

    "Remaining Invalid Rows":[

        (order_payments["payment_value"] <= 0).sum(),

        (order_payments["payment_installments"] < 1).sum(),

        (products["product_weight_g"] <= 0).sum()

    ]

})

validation

,Check,Remaining Invalid Rows
0,Payment Value > 0,0
1,Payment Installments >= 1,0
2,Product Weight > 0,0


# Text Standardization

The localized datasets were inspected for text consistency.

Since all customer cities, seller cities, regions, payment methods, product categories, and carrier names were generated through a controlled localization pipeline, no additional text normalization was required.

Operations such as trimming whitespace, correcting capitalization, or standardizing spelling were therefore deemed unnecessary.

In [20]:
text_validation = pd.DataFrame({

    "Dataset": [
        "Customers",
        "Sellers",
        "Products",
        "Payments"
    ],

    "Column": [
        "customer_city",
        "seller_city",
        "product_category_name",
        "payment_type"
    ],

    "Status": [
        "Already Standardized",
        "Already Standardized",
        "Already Standardized",
        "Already Standardized"
    ]

})

text_validation

,Dataset,Column,Status
0,Customers,customer_city,Already Standardized
1,Sellers,seller_city,Already Standardized
2,Products,product_category_name,Already Standardized
3,Payments,payment_type,Already Standardized


# Cleaning Summary

In [21]:
cleaning_summary = pd.DataFrame({

    "Cleaning Step": [

        "Datetime Conversion",
        "Duplicate Audit",
        "Missing Product Dimensions",
        "Missing Product Categories",
        "Logical Missing Values",
        "Invalid Payment Values",
        "Invalid Installments",
        "Invalid Product Weights",
        "Text Standardization"

    ],

    "Action": [

        "Converted",
        "Audited (No Removal)",
        "Filled with Median",
        "Filled with 'Unknown'",
        "Preserved",
        "Removed",
        "Removed",
        "Replaced with Median",
        "Already Standardized"

    ]

})

cleaning_summary

,Cleaning Step,Action
0,Datetime Conversion,Converted
1,Duplicate Audit,Audited (No Removal)
2,Missing Product Dimensions,Filled with Median
3,Missing Product Categories,Filled with 'Unknown'
4,Logical Missing Values,Preserved
5,Invalid Payment Values,Removed
6,Invalid Installments,Removed
7,Invalid Product Weights,Replaced with Median
8,Text Standardization,Already Standardized


# Feature Engineering

## Temporal Features

In [22]:
orders["purchase_year"] = (
    orders["order_purchase_timestamp"]
    .dt.year
)

orders["purchase_month"] = (
    orders["order_purchase_timestamp"]
    .dt.month
)

orders["purchase_month_name"] = (
    orders["order_purchase_timestamp"]
    .dt.month_name()
)

orders["purchase_quarter"] = (
    orders["order_purchase_timestamp"]
    .dt.quarter
)

orders["purchase_weekday"] = (
    orders["order_purchase_timestamp"]
    .dt.day_name()
)

orders["purchase_hour"] = (
    orders["order_purchase_timestamp"]
    .dt.hour
)

orders["is_weekend"] = (
    orders["purchase_weekday"]
    .isin(["Saturday", "Sunday"])
)

In [23]:
orders[
    [
        "order_purchase_timestamp",
        "purchase_year",
        "purchase_month",
        "purchase_month_name",
        "purchase_quarter",
        "purchase_weekday",
        "purchase_hour",
        "is_weekend"
    ]
].head()

,order_purchase_timestamp,purchase_year,purchase_month,purchase_month_name,purchase_quarter,purchase_weekday,purchase_hour,is_weekend
0,2025-10-02 10:56:33,2025,10,October,4,Thursday,10,False
1,2026-07-24 20:41:37,2026,7,July,3,Friday,20,False
2,2026-08-08 08:38:49,2026,8,August,3,Saturday,8,True
3,2025-11-18 19:28:06,2025,11,November,4,Tuesday,19,False
4,2026-02-13 21:18:39,2026,2,February,1,Friday,21,False


## Delivery Features

In [24]:
orders["shipping_time_days"] = (
    orders["order_delivered_carrier_date"]
    - orders["order_approved_at"]
).dt.days

In [25]:
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [26]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

In [29]:
orders["delivery_status"] = pd.NA

orders.loc[
    orders["delivery_delay_days"] < 0,
    "delivery_status"
] = "Early"

orders.loc[
    orders["delivery_delay_days"] == 0,
    "delivery_status"
] = "On Time"

orders.loc[
    orders["delivery_delay_days"] > 0,
    "delivery_status"
] = "Late"

In [30]:
orders[
    [
        "shipping_time_days",
        "delivery_time_days",
        "delivery_delay_days",
        "delivery_status"
    ]
].head()

,shipping_time_days,delivery_time_days,delivery_delay_days,delivery_status
0,2.0,8.0,-8.0,Early
1,0.0,13.0,-6.0,Early
2,0.0,9.0,-18.0,Early
3,3.0,13.0,-13.0,Early
4,0.0,2.0,-10.0,Early


In [31]:
orders["delivery_status"].value_counts(dropna=False)

delivery_status
Early      88649
Late        6535
<NA>        2965
On Time     1292
Name: count, dtype: int64

## Payment Features

In [32]:
order_payments["cod_flag"] = (
    order_payments["payment_type"] == "Cash on Delivery"
)

In [33]:
payment_bins = [
    0,
    200,
    500,
    1000,
    np.inf
]

payment_labels = [
    "Low",
    "Medium",
    "High",
    "Premium"
]

order_payments["payment_value_category"] = pd.cut(
    order_payments["payment_value"],
    bins=payment_bins,
    labels=payment_labels,
    include_lowest=True
)

In [34]:
installment_bins = [
    0,
    1,
    3,
    6,
    np.inf
]

installment_labels = [
    "1",
    "2-3",
    "4-6",
    "7+"
]

order_payments["payment_installments_bucket"] = pd.cut(
    order_payments["payment_installments"],
    bins=installment_bins,
    labels=installment_labels
)

In [35]:
order_payments[
    [
        "payment_type",
        "payment_value",
        "payment_value_category",
        "payment_installments",
        "payment_installments_bucket",
        "cod_flag"
    ]
].head(10)

,payment_type,payment_value,payment_value_category,payment_installments,payment_installments_bucket,cod_flag
0,Bank Card,181.77,Low,8,7+,False
1,Bank Card,44.63,Low,1,1,False
2,Bank Card,120.25,Low,1,1,False
3,Bank Card,197.24,Low,8,7+,False
4,Bank Card,235.06,Medium,2,2-3,False
5,Bank Card,175.90,Low,2,2-3,False
6,Bank Card,148.52,Low,1,1,False
7,Bank Card,94.87,Low,3,2-3,False
8,Bank Card,624.19,High,6,4-6,False
9,Cash on Delivery,95.07,Low,1,1,True


In [36]:
print(order_payments["payment_value_category"].value_counts())

print()

print(order_payments["payment_installments_bucket"].value_counts())

print()

print(order_payments["cod_flag"].value_counts())

payment_value_category
Low        56438
Medium     35629
High        8090
Premium     3718
Name: count, dtype: int64

payment_installments_bucket
1      52537
2-3    22874
4-6    16257
7+     12207
Name: count, dtype: int64

cod_flag
False    84091
True     19784
Name: count, dtype: int64


## Product Features

In [37]:
products["product_volume_cm3"] = (
    products["product_length_cm"]
    * products["product_height_cm"]
    * products["product_width_cm"]
)

In [38]:
products[
    [
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_volume_cm3"
    ]
].head()

,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3
0,16.0,10.0,14.0,2240.0
1,30.0,18.0,20.0,10800.0
2,18.0,9.0,15.0,2430.0
3,26.0,4.0,26.0,2704.0
4,20.0,17.0,13.0,4420.0


In [39]:
products["product_volume_cm3"].describe()

count     32951.000000
mean      16563.485843
std       27056.334089
min         168.000000
25%        2880.000000
50%        6840.000000
75%       18480.000000
max      296208.000000
Name: product_volume_cm3, dtype: float64

## Review Features

In [40]:
order_reviews["review_sentiment"] = pd.NA

order_reviews.loc[
    order_reviews["review_score"].isin([1, 2]),
    "review_sentiment"
] = "Negative"

order_reviews.loc[
    order_reviews["review_score"] == 3,
    "review_sentiment"
] = "Neutral"

order_reviews.loc[
    order_reviews["review_score"].isin([4, 5]),
    "review_sentiment"
] = "Positive"

In [41]:
order_reviews["would_recommend"] = (
    order_reviews["review_score"] >= 4
)

In [42]:
order_reviews[
    [
        "review_score",
        "review_sentiment",
        "would_recommend"
    ]
].head(10)

,review_score,review_sentiment,would_recommend
0,4,Positive,True
1,5,Positive,True
2,5,Positive,True
3,5,Positive,True
4,5,Positive,True
5,1,Negative,False
6,5,Positive,True
7,5,Positive,True
8,5,Positive,True
9,4,Positive,True


In [43]:
print(order_reviews["review_sentiment"].value_counts())

print()

print(order_reviews["would_recommend"].value_counts())

review_sentiment
Positive    76470
Negative    14575
Neutral      8179
Name: count, dtype: int64

would_recommend
True     76470
False    22754
Name: count, dtype: int64


# Export Cleaned Datasets

In [44]:
customers.to_csv(
    OUTPUT_PATH / "chrily_customers.csv",
    index=False
)

geolocation.to_csv(
    OUTPUT_PATH / "chrily_geolocation.csv",
    index=False
)

order_items.to_csv(
    OUTPUT_PATH / "chrily_order_items.csv",
    index=False
)

order_payments.to_csv(
    OUTPUT_PATH / "chrily_order_payments.csv",
    index=False
)

order_reviews.to_csv(
    OUTPUT_PATH / "chrily_order_reviews.csv",
    index=False
)

orders.to_csv(
    OUTPUT_PATH / "chrily_orders.csv",
    index=False
)

products.to_csv(
    OUTPUT_PATH / "chrily_products.csv",
    index=False
)

sellers.to_csv(
    OUTPUT_PATH / "chrily_sellers.csv",
    index=False
)

category_translation.to_csv(
    OUTPUT_PATH / "chrily_category_translation.csv",
    index=False
)

print("✓ Cleaned datasets exported successfully.")

✓ Cleaned datasets exported successfully.


In [45]:
for file in sorted(OUTPUT_PATH.glob("*.csv")):

    df = pd.read_csv(file)

    print("=" * 70)
    print(file.name)
    print(f"Rows    : {len(df):,}")
    print(f"Columns : {df.shape[1]}")

chrily_category_translation.csv
Rows    : 71
Columns : 2
chrily_customers.csv
Rows    : 99,441
Columns : 6
chrily_geolocation.csv
Rows    : 1,000,163
Columns : 5
chrily_order_items.csv
Rows    : 112,650
Columns : 9
chrily_order_payments.csv
Rows    : 103,875
Columns : 8
chrily_order_reviews.csv
Rows    : 99,224
Columns : 9
chrily_orders.csv
Rows    : 99,441
Columns : 25
chrily_products.csv
Rows    : 32,951
Columns : 10
chrily_sellers.csv
Rows    : 3,095
Columns : 4
